In [201]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from dotenv import load_dotenv
from tqdm import tqdm

from eval_utils import vec_ser_evidence_for_sequence_in_file, extract_topk_sequences_from_evidence, text_topk_of_chords_string_in_file
from graph_utils import graph_from_string

from langchain_ollama import ChatOllama
from langchain.tools import tool

from ollama import chat
from ollama import ChatResponse

# Load environment variables from .env file
load_dotenv()

True

In [202]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if f.endswith( ('.mid', '.midi', '.mxl', '.xml', '.musicxml') ):
                file_names.append(f)
                file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths
# end absoluteFilePaths

hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))
gjt_file_names, gjt_file_paths = absoluteFilePaths(os.getenv('VAL_GJT'))

device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [203]:
# in_seq = 'b_G:7_@1;C#:7_@1;C:maj_@2'
# in_seq = 'b_D:min7_@1;G:7_@1;C:maj_@2'
in_seq = 'b_D#:7_@1;D:7_@1;C#:7_@1;C:maj7_@1'
# in_seq = 'b_D#:maj13_@1;D:7_@1;C#:7(b13)_@1;C:maj13_@1'

In [204]:
g = graph_from_string(in_seq)

In [205]:
g.print_info()

Number of bars: 1
Segment bar range: [0, 1)
Segment graph features:
HeteroData(
  pitch={ x=[12, 12] },
  event={
    num_nodes=4,
    x=[4, 1],
  },
  (pitch, participates, event)={
    edge_index=[2, 16],
    edge_attr=[16, 5],
  },
  (event, next, event)={
    edge_index=[2, 3],
    edge_attr=[3, 6],
  }
)
Segment graph bars:
Bar 1:
Bar token positions: [2, 2, 2, 2, 2, 2, 2, 2]
Number of chord objects in bar: 4
Chord object 1:
Chord label: D#:7
Pitch classes: [1, 3, 7, 10]
Root: 3
Chord ID: 100
Bar Positions: [0, 1]
Token Positions: [2, 2]
Graph Features:
tensor([[0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.]])
BiLSTM Features
tensor([0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0.])
Chord object 2:
Chord label: D:7
Pitch classes: [0, 2, 6, 9]
Root: 2
Chord ID: 71
Bar Positions: [2, 3]
Token Positions: [2, 2]
Graph Features:
tensor([[0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
       

In [206]:
seg = g.segment_graph

pitch_list = seg["pitch"].x.tolist()
event_list = seg["event"].x.squeeze(-1).tolist()

participates_index = seg["pitch", "participates", "event"].edge_index
participates_attr = seg["pitch", "participates", "event"].edge_attr
pitch_participates = [
    (int(u), int(v), attr.tolist())
    for u, v, attr in zip(
        participates_index[0],
        participates_index[1],
        participates_attr
    )
]

next_index = seg["event", "next", "event"].edge_index
next_attr = seg["event", "next", "event"].edge_attr
event_next = [
    (int(u), int(v), attr.tolist())
    for u, v, attr in zip(
        next_index[0],
        next_index[1],
        next_attr
    )
]

# previous_root_retention,
# current_root_retention,
# common_pitch_class_ratio,
# upward_semitone_resolution_to_root,
# downward_semitone_resolution_to_root,
# descending_fifth_root_motion
transition_properties = [
    'previous root retention',
    'current root retention',
    'common pitch class ration',
    'upward semitone resolution to root',
    'downward semitone resolution to root',
    'descending fifth root motion'
]

In [207]:
# print(participates_index)
# print(participates_attr)
# print(next_index)
# print(next_attr)

print(len(pitch_participates))
print(len(event_next))

16
3


In [208]:
print(event_next[0])

(0, 1, [0.0, 0.0, 0.0, 1.0, 1.0, 0.0])


In [209]:
transition_counter = 0
transition_descriptions = []
for ne in event_next:
    transition_counter += 1
    transition_description = f'Transition {transition_counter} has: '
    transition_description += f'{ne[2][2]} of pitch classes between chords retained'
    for i, attrs in enumerate(ne[2]):
        if attrs == 1:
            transition_description += ', ' + transition_properties[i]
    transition_description += '. '
    transition_descriptions.append(transition_description)

In [210]:
for td in transition_descriptions:
    print(td)

Transition 1 has: 0.0 of pitch classes between chords retained, upward semitone resolution to root, downward semitone resolution to root. 
Transition 2 has: 0.0 of pitch classes between chords retained, upward semitone resolution to root, downward semitone resolution to root. 
Transition 3 has: 0.1428571492433548 of pitch classes between chords retained, upward semitone resolution to root, downward semitone resolution to root. 


In [211]:
y = graph_adapter_model(seg)

In [212]:
print(y)

tensor([-2.3739e-01,  1.8117e-01,  8.8747e-02,  2.7542e-01, -2.5755e-01,
        -2.7376e-01,  8.6209e-02,  2.4831e-01, -8.2213e-02, -2.6330e-01,
        -2.3921e-01, -1.3045e-02,  3.2416e-01,  1.9970e-01,  1.7948e-01,
        -2.3956e-02, -6.8115e-02, -3.7143e-02, -2.0507e-01, -1.5693e-01,
         3.5116e-02,  6.1551e-02,  2.3594e-01, -1.3815e-02,  3.1802e-01,
         1.9607e-01, -3.2423e-01,  4.3358e-01,  3.0546e-01,  1.4036e-01,
        -1.1516e-01,  1.3235e-01, -1.9533e-01,  1.7004e-01,  2.4015e-01,
         6.7490e-02, -4.0673e-01, -1.9524e-01,  2.6998e-01, -1.3838e-01,
         2.1887e-01,  2.4217e-01, -2.1338e-01, -1.0564e-01, -2.4640e-02,
         2.0923e-01, -1.9718e-01,  1.6707e-01, -7.9788e-02,  1.5660e-02,
        -1.1252e-01, -5.9694e-02,  9.5257e-02, -2.0019e-01,  1.2583e-01,
        -2.6417e-01,  2.9413e-03,  3.0989e-02,  3.2652e-01, -6.7419e-02,
        -5.3582e-02, -4.5413e-02, -4.2516e-02, -6.6821e-02,  2.3579e-01,
         2.4520e-02,  1.3386e-01,  6.2837e-02, -1.1

In [273]:
file_path = gjt_file_paths[4]
# file_path = gjt_file_paths[0] # query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 8: ['A:7', 'D:7', 'G:maj6', 'C:maj6'],

In [274]:
bars_string, evidence = vec_ser_evidence_for_sequence_in_file(
    in_seq,
    file_path,
    tokenizer,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    max_seq_len=16
)

In [275]:
print(evidence[1]['graph'])
print(evidence[2]['graph'])
print(evidence[3]['graph'])

[0.26300162 0.12775816 0.10801214 0.10184374 0.04189643 0.04189641
 0.17400385 0.11811481 0.26932508 0.02681946 0.02681946 0.22792619
 0.13663635 0.22792619 0.13663635 0.22792619]
[0.22949851 0.17680874 0.22546215 0.21059774 0.13181779 0.30009574
 0.26932508 0.29477003 0.28991565 0.13074273 0.15971707 0.23715627
 0.23746344 0.2371563  0.23746347]
[0.24179572 0.20487107 0.09650765 0.17528707 0.30885291 0.33055133
 0.26977262 0.31688669 0.25095934 0.16545695 0.13345185 0.14626461
 0.18394339 0.14626461]


In [276]:
print(evidence[1]['token'])
print(evidence[2]['token'])
print(evidence[3]['token'])

[-0.06660563 -0.09628499  0.30568817  0.18182841  0.20700222  0.20700222
  0.11001594  0.08943614  0.19732031  0.03737637  0.03737637  0.09818552
  0.29149088  0.09818552  0.29149088  0.09818552]
[-0.13624662  0.12155181  0.33738098  0.27715331  0.15197891  0.06203331
  0.19732031  0.06395925  0.20058082  0.03595387  0.01947649  0.11460888
  0.35114196  0.11460888  0.35114196]
[ 0.02961232  0.1485313   0.3311497   0.22950435 -0.04638936  0.0730302
  0.08200905  0.21396133  0.21098515 -0.06589966  0.1008281   0.11677696
  0.33763701  0.11677696]


In [277]:
print(evidence[1]['adapter'])
print(evidence[2]['adapter'])
print(evidence[3]['adapter'])

[0.27617243 0.12438454 0.21222535 0.24006876 0.2181077  0.21810767
 0.2080902  0.26187432 0.3558225  0.163607   0.163607   0.26726151
 0.37189001 0.26726151 0.37189001 0.26726151]
[0.25412434 0.37631321 0.48977605 0.45930034 0.29639    0.36296731
 0.3558225  0.24245001 0.39675918 0.37453893 0.33305693 0.34011066
 0.47236431 0.34011069 0.47236431]
[0.3866027  0.3956427  0.38591355 0.36952126 0.21872157 0.29156193
 0.18054217 0.4280656  0.368765   0.33526361 0.38824585 0.28844419
 0.44905362 0.28844422]


In [278]:
print(evidence[2]['chord_symbols'])

[['C:maj7', 'F:9'], ['F:9', 'A#:9'], ['A#:9', 'A:7'], ['A:7', 'D:9'], ['D:9', 'D:9'], ['D:9', 'D:min7'], ['D:min7', 'G:7'], ['G:7', 'D:min7', 'G:7'], ['D:min7', 'G:7', 'C:maj6'], ['C:maj6', 'C:maj6'], ['C:maj6', 'B:hdim7', 'E:7'], ['B:hdim7', 'E:7', 'A:min6'], ['A:min6', 'B:hdim7', 'E:7'], ['B:hdim7', 'E:7', 'A:min6'], ['A:min6', 'B:hdim7', 'E:7']]


In [279]:
topk_per_model = extract_topk_sequences_from_evidence(evidence, 5)

print(topk_per_model['graph'])

{'similarities': [0.3305513262748718, 0.3257427513599396, 0.3191637694835663, 0.31688669323921204, 0.314561128616333], 'chord_symbols': [['D:9', 'D:min7', 'G:7'], ['D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6'], ['D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6'], ['G:7', 'D:min7', 'G:7', 'C:maj6'], ['D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6']], 'starting_bars': [5, 5, 4, 7, 6], 'bar_lengths': [3, 5, 6, 3, 4]}


In [280]:
for k, v in topk_per_model['graph'].items():
    print(k, ': ', v)

similarities :  [0.3305513262748718, 0.3257427513599396, 0.3191637694835663, 0.31688669323921204, 0.314561128616333]
chord_symbols :  [['D:9', 'D:min7', 'G:7'], ['D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6'], ['D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6'], ['G:7', 'D:min7', 'G:7', 'C:maj6'], ['D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6']]
starting_bars :  [5, 5, 4, 7, 6]
bar_lengths :  [3, 5, 6, 3, 4]


In [281]:
bars_string, in_seq_list, text_descriptions = text_topk_of_chords_string_in_file(
    in_seq,
    tokenizer,
    file_path,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    max_seq_len=16,
    k=5
)

In [282]:
print(in_seq_list)

['D#:7', 'D:7', 'C#:7', 'C:maj7']


In [283]:
print(bars_string)

Piece:
bar 0: C:maj7 
bar 1: F:9 
bar 2: A#:9 
bar 3: A:7 
bar 4: D:9 
bar 5: D:9 
bar 6: D:min7 
bar 7: G:7 
bar 8: D:min7 G:7 
bar 9: C:maj6 
bar 10: C:maj6 
bar 11: B:hdim7 E:7 
bar 12: A:min6 
bar 13: B:hdim7 E:7 
bar 14: A:min6 
bar 15: B:hdim7 E:7 



In [284]:
print(text_descriptions)

{'token': ["query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 0 - 10: ['C:maj7', 'F:9', 'A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6', 'C:maj6'],  with similarity: 0.3572123646736145", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 2 - 10: ['A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6', 'C:maj6'],  with similarity: 0.35648632049560547", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 1 - 10: ['F:9', 'A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6', 'C:maj6'],  with similarity: 0.3540080487728119", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 14 - 15: ['A:min6', 'B:hdim7', 'E:7'],  with similarity: 0.35114195942878723", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 2 - 9: ['A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6'],  with similarity: 0.347911536693573"], 'graph': ["query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 5 - 7: ['D

In [285]:
print(bars_string)

Piece:
bar 0: C:maj7 
bar 1: F:9 
bar 2: A#:9 
bar 3: A:7 
bar 4: D:9 
bar 5: D:9 
bar 6: D:min7 
bar 7: G:7 
bar 8: D:min7 G:7 
bar 9: C:maj6 
bar 10: C:maj6 
bar 11: B:hdim7 E:7 
bar 12: A:min6 
bar 13: B:hdim7 E:7 
bar 14: A:min6 
bar 15: B:hdim7 E:7 



In [286]:
string_descriptions = {
    'token': '',
    'graph': '',
    'adapter': ''
}

for k , v in text_descriptions.items():
    print(k)
    for t in v:
        print(t)
        string_descriptions[k] += t + '\n'

token
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 0 - 10: ['C:maj7', 'F:9', 'A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6', 'C:maj6'],  with similarity: 0.3572123646736145
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 2 - 10: ['A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6', 'C:maj6'],  with similarity: 0.35648632049560547
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 1 - 10: ['F:9', 'A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6', 'C:maj6'],  with similarity: 0.3540080487728119
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 14 - 15: ['A:min6', 'B:hdim7', 'E:7'],  with similarity: 0.35114195942878723
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 2 - 9: ['A#:9', 'A:7', 'D:9', 'D:9', 'D:min7', 'G:7', 'D:min7', 'G:7', 'C:maj6'],  with similarity: 0.347911536693573
graph
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 5 - 7: ['D:9', 'D:min7', 'G:7'],  wi

In [287]:
# central_prompt = '''
# You are a music harmony expert. You will be give the chord sequence of a piece, per bar.
# You will also be given a query sequence of chords.
# You job is to identify chord subsequences within the piece that are similar 
# (not necessarily identical) to the query sequence. You need to provide details about the 
# reasons why you believe these segments are similar to the query.\n\n
# '''

# focus_properties_prompt = f'''
# You can focus your similarity criteria toward identifying similarities per transition
# in the query and the piece sequences based on the following criteria between pairs
# of chords in each transition:
# {transition_properties}\n\n
# '''

# piece_and_query_prompt = f'''
# Here is the piece sequence:\n
# {bars_string}\n\n
# Here is the query:\n
# {in_seq_list}
# '''

# adapter_tool_prompt = '''
# You can use the output of a model that assessed the similarities (maximum 1, minimum -1)
# between bar segments of the pieces and query:
# ''' + string_descriptions['adapter']

In [288]:
# print(central_prompt + focus_properties_prompt + piece_and_query_prompt )

In [289]:
# model_name = 'deepseek-r1:14b'
# model_name = 'qwen2.5-coder:14b'

In [290]:
# response: ChatResponse = chat(
#   model=model_name,
#   messages=[
#     {
#       'role': 'user',
#       'content': central_prompt + focus_properties_prompt + piece_and_query_prompt,
#     }
#   ],
#   keep_alive=0
# )
# basic_response = response['message']['content']
# print(basic_response)

In [291]:
# print(central_prompt + focus_properties_prompt + piece_and_query_prompt + adapter_tool_prompt)

In [292]:
# response: ChatResponse = chat(
#   model=model_name,
#   messages=[
#     {
#       'role': 'user',
#       'content': central_prompt + focus_properties_prompt + piece_and_query_prompt + adapter_tool_prompt,
#     }
#   ],
#   keep_alive=0
# )
# basic_response = response['message']['content']
# print(basic_response)